# SautiCivic Bridge — Cloud ASR Benchmark Runner (Google Colab)

This notebook runs the **OpenAI Whisper large-v3** (and optionally Deepgram Nova-3 and Sahara v2.5) transcription benchmarks on Google Colab GPU without exhausting your local laptop memory/RAM.

### Recommended Colab Setup:
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or A100 if available) as the Hardware Accelerator.
3. Run the cells step-by-step.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Set Up Repository & Corpus

Choose **Option A** (Clone from GitHub) OR **Option B** (Upload repository zip).

In [ ]:
# Option A: Clone from GitHub (Replace with your repo URL if public/authenticated)
# !git clone https://github.com/your-username/Sauticivic.git
# %cd Sauticivic

# Option B: Upload Sauticivic zip directly if running from local workspace
import os
from pathlib import Path

if not os.path.exists("bench"): 
    print("Please upload your Sauticivic folder or clone the repository.")
else:
    print("Found repository root. Ready to proceed!")

## 3. Install Dependencies
Install `openai-whisper`, `ffmpeg`, and `deepgram-sdk`.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q openai-whisper deepgram-sdk==3.11.0 pydantic pydantic-settings httpx

## 4. Run OpenAI Whisper large-v3 Benchmark
Transcribes the 30 Tier A recorded audio clips using GPU-accelerated Whisper `large-v3`.

In [ ]:
!python3 bench/models/run_whisper.py \
    --corpus bench/corpus/tier_a_recorded/audio \
    --output-dir bench/results/transcripts \
    --model-size large-v3

## 5. (Optional) Run Deepgram Nova-3 Benchmark on Cloud

In [ ]:
# Set your Deepgram API key
import os
os.environ["DEEPGRAM_API_KEY"] = "YOUR_DEEPGRAM_API_KEY"  # replace with your key

!python3 bench/models/run_deepgram.py \
    --corpus bench/corpus/tier_a_recorded/audio \
    --output-dir bench/results/transcripts \
    --model nova-3

## 6. (Optional) Run Sahara v2.5 Benchmark on Cloud

In [ ]:
# Set your Sahara API key
import os
os.environ["SAHARA_API_KEY"] = "YOUR_SAHARA_API_KEY"  # replace with your key

!python3 bench/models/run_sahara.py \
    --corpus bench/corpus/tier_a_recorded/audio \
    --output-dir bench/results/transcripts

## 7. Package and Download Transcripts
Downloads all generated transcript JSON files in a zip archive so you can extract them into `bench/results/transcripts/` in your local project.

In [ ]:
!zip -r transcripts_cloud_results.zip bench/results/transcripts/

from google.colab import files
files.download("transcripts_cloud_results.zip")

### Next Steps on Local Machine:
1. Extract `transcripts_cloud_results.zip` into `bench/results/transcripts/`.
2. Run the full benchmark metrics suite:
   ```bash
   PYTHONPATH=backend:. python3 -m bench.metrics.run_full_benchmark
   ```
3. View the generated `v4_results.json` and updated metrics.